In [16]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_DASHBOARD = PROJECT_ROOT / "data" / "dashboard"
DATA_DASHBOARD.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("DATA_DASHBOARD:", DATA_DASHBOARD)


PROJECT_ROOT: C:\Users\User\Documents\retailmind-prototype
DATA_PROCESSED: C:\Users\User\Documents\retailmind-prototype\data\processed
DATA_DASHBOARD: C:\Users\User\Documents\retailmind-prototype\data\dashboard


In [17]:
turns_path = DATA_PROCESSED / "uss_english_issue_tagged_llm_subset.parquet"
topics_row_path = DATA_PROCESSED / "uss_english_topics_rows_llm_subset.parquet"
topics_summary_path = DATA_PROCESSED / "uss_english_topics_summary_llm_subset.parquet"
repairs_path = DATA_PROCESSED / "uss_english_prompt_repairs_llm_subset.parquet"


df_turns = pd.read_parquet(turns_path)
df_topics_rows = pd.read_parquet(topics_row_path)
df_topics_summary = pd.read_parquet(topics_summary_path)
df_repairs = pd.read_parquet(repairs_path)

print("df_turns:", df_turns.shape)
print("df_topics_rows:", df_topics_rows.shape)
print("df_topics_summary:", df_topics_summary.shape)
print("df_repairs:", df_repairs.shape)

# Quick check: turns are full dataset; topics/repairs are only for the clustered subset
display(df_turns.tail(3))
display(df_topics_summary.head(5))
display(df_repairs.head(5))


df_turns: (1340, 21)
df_topics_rows: (132, 5)
df_topics_summary: (6, 6)
df_repairs: (6, 7)


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label,...,low_satisfaction,entity_tag,system_text_only,last_system_text,satisfaction_score,satisfaction_reason,satisfaction_source,issues,severity,reason
1337,998,33,USER,"No, thanks, It's all",THANK_YOU,"3,3,4","[3, 3, 4]",False,3.333333,THANK_YOU,...,False,None,None,Do you need anything else?,5.0,The user expressed complete satisfaction by in...,llm,[],NONE,
1338,998,34,SYSTEM,Have a nice day!,GOODBYE,None,None,False,NaN,GOODBYE,...,False,None,Have a nice day!,Have a nice day!,NaN,None,None,[],NONE,
1339,998,35,USER,OVERALL,None,"3,3,4","[3, 3, 4]",True,3.333333,None,...,False,None,None,Have a nice day!,NaN,None,None,[],NONE,


,topic_id,topic_label,n_examples,top_issues,example_texts,example_reason
0,1,Topic 1: MISSING_CONTEXT / TONE_ISSUE,31,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[Probably not., Okay., Nope.]","The bot's question lacked context, making it d..."
1,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,25,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]",[It was okay. Too many Too many special effect...,"The bot's question lacked context, failing to ..."
2,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT,25,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FACT]",[Great! Can you tell me if I have an available...,The bot failed to provide context regarding th...
3,4,Topic 4: UNSUPPORTED_INTENT / MISSING_CONTEXT,22,"[UNSUPPORTED_INTENT, MISSING_CONTEXT, SUCCESS_...","[Hello! Can you help me find an apartment?, Th...",The bot did not have any previous context to u...
4,5,Topic 5: MISSING_CONTEXT / TONE_ISSUE,21,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[then effects, [inaudible], first time in the ...","The bot's question lacks context, making it di..."


,topic_id,topic_label,root_cause,suggested_prompt_changes,system_prompt_snippet,guardrail_rules,evaluation_checks
0,1,Topic 1: MISSING_CONTEXT / TONE_ISSUE,The bot's questions lack sufficient context an...,[Incorporate more context in the bot's questio...,Ensure that the bot provides contextually rele...,[Always provide sufficient context before aski...,[Review user interactions for instances of vag...
1,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,The bot's responses lack context and do not ad...,[Incorporate follow-up questions that ask for ...,"When responding to user feedback about movies,...",[Always acknowledge the user's feedback and ex...,[Assess if the bot's responses include context...
2,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT,The virtual assistant lacks the ability to mai...,[Implement a context management system to reta...,You are a virtual assistant that understands u...,[Ensure that the assistant confirms the user's...,[Test the assistant's ability to retain contex...
3,4,Topic 4: UNSUPPORTED_INTENT / MISSING_CONTEXT,The virtual assistant lacks the ability to han...,[Implement a context management system to reta...,You are a virtual assistant designed to help u...,"[If a user query contains unsupported intents,...",[Test the assistant's ability to retain contex...
4,5,Topic 5: MISSING_CONTEXT / TONE_ISSUE,The bot's responses lack sufficient context an...,[Incorporate clarifying questions to gather ne...,Ensure that responses are contextually relevan...,[Always ask clarifying questions if the user's...,[Review user interactions for instances of con...


In [18]:
required_turns_cols = {"dataset", "conv_id", "turn_id", "speaker", "text", "low_satisfaction", "issues", "severity", "reason"}
missing = required_turns_cols - set(df_turns.columns)
if missing:
    raise ValueError(f"df_turns missing columns: {missing}")

required_topics_rows_cols = {"dataset", "conv_id", "turn_id", "topic_id"}
missing = required_topics_rows_cols - set(df_topics_rows.columns)
if missing:
    raise ValueError(f"df_topics_rows missing columns: {missing}")

required_repairs_cols = {"topic_id", "topic_label", "root_cause", "suggested_prompt_changes", "system_prompt_snippet", "guardrail_rules", "evaluation_checks"}
missing = required_repairs_cols - set(df_repairs.columns)
if missing:
    raise ValueError(f"df_repairs missing columns: {missing}")

print("Column checks passed.")


Column checks passed.


In [19]:
def normalize_list_field(val):
    if isinstance(val, list):
        return [str(x) for x in val]
    if isinstance(val, np.ndarray):
        return [str(x) for x in val.tolist()]
    if isinstance(val, str):
        # treat as a single entry unless it looks like a list literal
        val = val.strip()
        if val.startswith("[") and val.endswith("]"):
            try:
                import ast
                parsed = ast.literal_eval(val)
                if isinstance(parsed, list):
                    return [str(x) for x in parsed]
            except Exception:
                pass
        return [val] if val else []
    if pd.isna(val):
        return []
    return [str(val)]

# Normalize issues in turns
df_turns["issues"] = df_turns["issues"].apply(normalize_list_field)

# Normalize list fields in repairs
for col in ["suggested_prompt_changes", "guardrail_rules", "evaluation_checks"]:
    df_repairs[col] = df_repairs[col].apply(normalize_list_field)

print("Normalization done.")
df_turns[["issues"]].tail(10)


Normalization done.


,issues
1330,[]
1331,[]
1332,[]
1333,[]
1334,[]
1335,[]
1336,[]
1337,[]
1338,[]
1339,[]


In [20]:
# ===== Enrich turns with topic_id (topics exist only for clustered subset) =====

# Ensure merge keys have consistent dtypes (prevents silent no-match merges)
for col in ["conv_id", "turn_id"]:
    df_turns[col] = pd.to_numeric(df_turns[col], errors="coerce").astype("Int64")
    df_topics_rows[col] = pd.to_numeric(df_topics_rows[col], errors="coerce").astype("Int64")

df_turns["dataset"] = df_turns["dataset"].astype(str)
df_topics_rows["dataset"] = df_topics_rows["dataset"].astype(str)

df_turns_enriched = df_turns.merge(
    df_topics_rows[["dataset", "conv_id", "turn_id", "topic_id"]],
    on=["dataset", "conv_id", "turn_id"],
    how="left",
)

# Fill non-clustered turns with a sentinel topic_id
# (dashboard can treat -1 as "UNCLUSTERED")
df_turns_enriched["topic_id"] = (
    pd.to_numeric(df_turns_enriched["topic_id"], errors="coerce")
    .fillna(-1)
    .astype(int)
)

print("After topic_id merge:", df_turns_enriched.shape)
print("Turns with assigned topic_id (>=0):", (df_turns_enriched["topic_id"] >= 0).sum())
display(df_turns_enriched[["dataset", "conv_id", "turn_id", "speaker", "topic_id"]].head(10))


After topic_id merge: (1340, 22)
Turns with assigned topic_id (>=0): 132


,dataset,conv_id,turn_id,speaker,topic_id
0,CCPE,41,1,SYSTEM,-1
1,CCPE,41,2,USER,-1
2,CCPE,41,3,SYSTEM,-1
3,CCPE,41,4,USER,-1
4,CCPE,41,5,USER,5
5,CCPE,41,6,SYSTEM,-1
6,CCPE,41,7,USER,-1
7,CCPE,41,8,USER,-1
8,CCPE,41,9,SYSTEM,-1
9,CCPE,41,10,USER,-1


In [21]:
# ===== Attach topic_label to each turn (based on df_repairs) =====

# Build a mapping from topic_id -> topic_label
topic_label_map = (
    df_repairs[["topic_id", "topic_label"]]
    .dropna(subset=["topic_id"])
    .assign(topic_id=lambda d: pd.to_numeric(d["topic_id"], errors="coerce").astype(int))
    .drop_duplicates(subset=["topic_id"])
    .set_index("topic_id")["topic_label"]
    .to_dict()
)

df_turns_enriched["topic_label"] = df_turns_enriched["topic_id"].map(topic_label_map)

# Label non-clustered turns explicitly (better for dashboard filters than empty string)
df_turns_enriched.loc[df_turns_enriched["topic_id"] < 0, "topic_label"] = "UNCLUSTERED"
df_turns_enriched["topic_label"] = df_turns_enriched["topic_label"].fillna("UNCLUSTERED")

print("Distinct topic labels (sample):")
display(df_turns_enriched[["topic_id", "topic_label"]].drop_duplicates().sort_values("topic_id").head(15))



Distinct topic labels (sample):


,topic_id,topic_label
0,-1,UNCLUSTERED
417,0,Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT
61,1,Topic 1: MISSING_CONTEXT / TONE_ISSUE
14,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE
617,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT
611,4,Topic 4: UNSUPPORTED_INTENT / MISSING_CONTEXT
4,5,Topic 5: MISSING_CONTEXT / TONE_ISSUE


In [22]:
# ===== Create a unified satisfaction_score (prefer LLM score if present) =====

# Detect potential LLM score columns (adjust names if your llm notebook used a different one)
LLM_SCORE_CANDIDATES = ["llm_satisfaction_score", "llm_score", "score"]

llm_col = next((c for c in LLM_SCORE_CANDIDATES if c in df_turns_enriched.columns), None)

if llm_col is not None:
    df_turns_enriched["satisfaction_score"] = pd.to_numeric(df_turns_enriched[llm_col], errors="coerce")
    df_turns_enriched["satisfaction_source"] = "llm"
elif "satisfaction_mean" in df_turns_enriched.columns:
    df_turns_enriched["satisfaction_score"] = pd.to_numeric(df_turns_enriched["satisfaction_mean"], errors="coerce")
    df_turns_enriched["satisfaction_source"] = "dataset"
else:
    df_turns_enriched["satisfaction_score"] = np.nan
    df_turns_enriched["satisfaction_source"] = "missing"

null_rate = df_turns_enriched["satisfaction_score"].isna().mean()
print("satisfaction_score null rate:", round(null_rate, 4))
print("satisfaction_source counts:")
print(df_turns_enriched["satisfaction_source"].value_counts(dropna=False))



satisfaction_score null rate: 0.4716
satisfaction_source counts:
satisfaction_source
dataset    1340
Name: count, dtype: int64


In [23]:
import numpy as np
import pandas as pd
import ast

# --- Helpers for JSON-safe normalization ---

def normalize_list_for_json(val):
    """
    Ensure list-like fields are JSON-serializable Python lists of strings.
    Handles list, np.ndarray, stringified list, NaN/None.
    """
    if val is None:
        return []
    if isinstance(val, float) and np.isnan(val):
        return []
    if isinstance(val, pd._libs.missing.NAType):
        return []

    if isinstance(val, list):
        return [str(x) for x in val]

    if isinstance(val, np.ndarray):
        return [str(x) for x in val.tolist()]

    if isinstance(val, str):
        s = val.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(x) for x in parsed]
            except Exception:
                pass
        return [s] if s else []

    # fallback: wrap in list
    return [str(val)]


def contains_ndarray(x):
    return isinstance(x, np.ndarray)


# --- Select columns for the dashboard turns export ---
turn_cols = [
    "dataset", "conv_id", "turn_id", "speaker", "text",
    "satisfaction_score", "low_satisfaction",
    "issues", "severity", "reason",
    "topic_id", "topic_label",
    "satisfaction_source",  # keep, since we created it (useful for dashboard)
]

# Keep only columns that actually exist (prevents KeyError if a column is missing)
turn_cols = [c for c in turn_cols if c in df_turns_enriched.columns]

df_turns_out = df_turns_enriched[turn_cols].copy()

# Normalize list-like column(s) that can break JSON
if "issues" in df_turns_out.columns:
    df_turns_out["issues"] = df_turns_out["issues"].apply(normalize_list_for_json)

# Replace pandas missing values with Python None (JSON-friendly)
df_turns_out = df_turns_out.where(pd.notnull(df_turns_out), None)

# Safety check: ensure no ndarray remains (should be empty now)
bad_cols = [c for c in df_turns_out.columns if df_turns_out[c].apply(contains_ndarray).any()]
print("Columns containing ndarray after normalization:", bad_cols)

# If any remain, show a few examples (should not happen)
for c in bad_cols:
    idx = df_turns_out[df_turns_out[c].apply(contains_ndarray)].index[:5]
    print(f"\nColumn: {c}")
    for i in idx:
        v = df_turns_out.at[i, c]
        print(" index:", i, " type:", type(v), " value preview:", v[:5] if hasattr(v, "__len__") else v)

print("df_turns_out shape:", df_turns_out.shape)
display(df_turns_out.head(3))


Columns containing ndarray after normalization: []
df_turns_out shape: (1340, 13)


,dataset,conv_id,turn_id,speaker,text,satisfaction_score,low_satisfaction,issues,severity,reason,topic_id,topic_label,satisfaction_source
0,CCPE,41,1,SYSTEM,what kind of movies do u generally like watching,NaN,False,[],NONE,,-1,UNCLUSTERED,dataset
1,CCPE,41,2,USER,I prefer horror movies myself and also some ac...,3.0,False,[],NONE,,-1,UNCLUSTERED,dataset
2,CCPE,41,3,SYSTEM,why do u like horror and action,NaN,False,[],NONE,,-1,UNCLUSTERED,dataset


In [24]:
import json

turns_out_path = DATA_DASHBOARD / "dashboard_turns.jsonl"

records = df_turns_out.to_dict(orient="records")

with open(turns_out_path, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")

print("Saved:", turns_out_path)
print("Rows:", len(records))


Saved: C:\Users\User\Documents\retailmind-prototype\data\dashboard\dashboard_turns.jsonl
Rows: 1340


In [25]:
print("df_repairs.columns:", list(df_repairs.columns))
print("df_topics_summary.columns:", list(df_topics_summary.columns))


df_repairs.columns: ['topic_id', 'topic_label', 'root_cause', 'suggested_prompt_changes', 'system_prompt_snippet', 'guardrail_rules', 'evaluation_checks']
df_topics_summary.columns: ['topic_id', 'topic_label', 'n_examples', 'top_issues', 'example_texts', 'example_reason']


In [26]:
import json
import pandas as pd

def require_cols(df: pd.DataFrame, cols, name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

def export_json(path, records, indent=2):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=indent, default=str)
    print("Saved:", path, "| records:", len(records))


In [27]:
# ===== Export repairs for dashboard =====

repairs_cols = [
    "topic_id", "topic_label", "root_cause",
    "suggested_prompt_changes", "system_prompt_snippet",
    "guardrail_rules", "evaluation_checks",
]
require_cols(df_repairs, repairs_cols, "df_repairs")

df_repairs_out = df_repairs[repairs_cols].copy()

# Ensure stable types
df_repairs_out["topic_id"] = pd.to_numeric(df_repairs_out["topic_id"], errors="coerce").fillna(-1).astype(int)
df_repairs_out["topic_label"] = df_repairs_out["topic_label"].fillna("").astype(str)
df_repairs_out["root_cause"] = df_repairs_out["root_cause"].fillna("").astype(str)
df_repairs_out["system_prompt_snippet"] = df_repairs_out["system_prompt_snippet"].fillna("").astype(str)

# Normalize list-like fields using the notebook's existing normalizer
for c in ["suggested_prompt_changes", "guardrail_rules", "evaluation_checks"]:
    df_repairs_out[c] = df_repairs_out[c].apply(normalize_list_field)

# Replace any remaining missing values with None for JSON
df_repairs_out = df_repairs_out.where(pd.notnull(df_repairs_out), None)

repairs_json_path = DATA_DASHBOARD / "dashboard_repairs.json"
export_json(repairs_json_path, df_repairs_out.to_dict(orient="records"))

display(df_repairs_out.head(2))


Saved: C:\Users\User\Documents\retailmind-prototype\data\dashboard\dashboard_repairs.json | records: 6


,topic_id,topic_label,root_cause,suggested_prompt_changes,system_prompt_snippet,guardrail_rules,evaluation_checks
0,1,Topic 1: MISSING_CONTEXT / TONE_ISSUE,The bot's questions lack sufficient context an...,[Incorporate more context in the bot's questio...,Ensure that the bot provides contextually rele...,[Always provide sufficient context before aski...,[Review user interactions for instances of vag...
1,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,The bot's responses lack context and do not ad...,[Incorporate follow-up questions that ask for ...,"When responding to user feedback about movies,...",[Always acknowledge the user's feedback and ex...,[Assess if the bot's responses include context...


In [28]:
# ===== Export topics summary for dashboard =====

topics_summary_cols = ["topic_id", "n_examples", "top_issues", "example_texts", "example_reason"]
require_cols(df_topics_summary, topics_summary_cols, "df_topics_summary")

df_topics_out = df_topics_summary.copy()

# Ensure stable types
df_topics_out["topic_id"] = pd.to_numeric(df_topics_out["topic_id"], errors="coerce").fillna(-1).astype(int)
df_topics_out["n_examples"] = pd.to_numeric(df_topics_out["n_examples"], errors="coerce").fillna(0).astype(int)
df_topics_out["example_reason"] = df_topics_out["example_reason"].fillna("").astype(str)

# Normalize list-like fields using existing normalizer
df_topics_out["top_issues"] = df_topics_out["top_issues"].apply(normalize_list_field)
df_topics_out["example_texts"] = df_topics_out["example_texts"].apply(normalize_list_field)

# Topic labels: take from df_repairs (authoritative for dashboard labels)
label_map = (
    df_repairs[["topic_id", "topic_label"]]
    .assign(topic_id=lambda d: pd.to_numeric(d["topic_id"], errors="coerce").fillna(-1).astype(int))
    .drop_duplicates(subset=["topic_id"])
    .set_index("topic_id")["topic_label"]
    .to_dict()
)

df_topics_out["topic_label"] = df_topics_out["topic_id"].map(label_map)
df_topics_out["topic_label"] = df_topics_out["topic_label"].fillna(df_topics_out["topic_id"].apply(lambda x: f"Topic {x}"))

# ---- Compute metrics from turn-level data (USER only, clustered topics only) ----
df_user = df_turns_enriched[df_turns_enriched["speaker"] == "USER"].copy()

# Choose low-satisfaction flag: prefer LLM flag if present
low_flag_col = "low_satisfaction_llm" if "low_satisfaction_llm" in df_user.columns else "low_satisfaction"
if low_flag_col not in df_user.columns:
    # If neither exists, create a missing column for safety
    df_user[low_flag_col] = np.nan

# Ensure numeric satisfaction_score exists (you created it earlier)
if "satisfaction_score" not in df_user.columns:
    df_user["satisfaction_score"] = np.nan

# Focus only on clustered topics (topic_id >= 0)
df_user_clustered = df_user[df_user["topic_id"] >= 0].copy()

topic_metrics = (
    df_user_clustered
    .groupby("topic_id", as_index=False)
    .agg(
        n_user_turns=("turn_id", "count"),
        low_satisfaction_rate=(low_flag_col, "mean"),
        avg_satisfaction=("satisfaction_score", "mean"),
    )
)

df_topics_out = df_topics_out.merge(topic_metrics, on="topic_id", how="left")

# Clean / fill
df_topics_out["n_user_turns"] = df_topics_out["n_user_turns"].fillna(0).astype(int)
df_topics_out["low_satisfaction_rate"] = df_topics_out["low_satisfaction_rate"].fillna(0.0)

# avg_satisfaction: keep None if missing (JSON-friendly)
df_topics_out["avg_satisfaction"] = df_topics_out["avg_satisfaction"].where(df_topics_out["avg_satisfaction"].notna(), None)

# Sort topics by biggest pain / importance
df_topics_out = df_topics_out.sort_values(["n_examples", "n_user_turns"], ascending=False)

# Replace remaining missing values with None for JSON export
df_topics_out = df_topics_out.where(pd.notnull(df_topics_out), None)

topics_json_path = DATA_DASHBOARD / "dashboard_topics.json"
export_json(topics_json_path, df_topics_out.to_dict(orient="records"))

display(df_topics_out.head(5))


Saved: C:\Users\User\Documents\retailmind-prototype\data\dashboard\dashboard_topics.json | records: 6


,topic_id,topic_label,n_examples,top_issues,example_texts,example_reason,n_user_turns,low_satisfaction_rate,avg_satisfaction
0,1,Topic 1: MISSING_CONTEXT / TONE_ISSUE,31,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[Probably not., Okay., Nope.]","The bot's question lacked context, making it d...",31,1.0,2.787634
1,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,25,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]",[It was okay. Too many Too many special effect...,"The bot's question lacked context, failing to ...",25,1.0,2.936000
2,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT,25,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FACT]",[Great! Can you tell me if I have an available...,The bot failed to provide context regarding th...,25,1.0,3.138000
3,4,Topic 4: UNSUPPORTED_INTENT / MISSING_CONTEXT,22,"[UNSUPPORTED_INTENT, MISSING_CONTEXT, SUCCESS_...","[Hello! Can you help me find an apartment?, Th...",The bot did not have any previous context to u...,22,1.0,3.095455
4,5,Topic 5: MISSING_CONTEXT / TONE_ISSUE,21,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[then effects, [inaudible], first time in the ...","The bot's question lacks context, making it di...",21,1.0,2.965079


In [29]:
# ===== Export "Sandbox cases" for the dashboard (optional but valuable) =====

BASELINE_SYSTEM_PROMPT = (
    "You are a helpful assistant. Be concise, accurate, and user-focused. "
    "If information is missing, ask clarifying questions. Avoid repeating the same question."
)

def conversation_snippet(df_all: pd.DataFrame, dataset: str, conv_id: int, max_turns: int = 14) -> str:
    convo = (
        df_all[(df_all["dataset"].astype(str) == str(dataset)) & (df_all["conv_id"] == conv_id)]
        .sort_values("turn_id")
        .head(max_turns)
    )

    lines = []
    for _, r in convo.iterrows():
        spk = str(r.get("speaker", "") or "")
        if spk == "USER":
            role = "User"
        elif spk == "SYSTEM":
            role = "System"
        else:
            role = "Bot"

        txt = r.get("text", "")
        txt = "" if txt is None else str(txt)
        lines.append(f"{role}: {txt}")

    return "\n".join(lines)

# Build conversation-level mapping: (dataset, conv_id) -> topic_id
clustered_convs = (
    df_topics_rows[["dataset", "conv_id", "topic_id"]]
    .drop_duplicates()
    .copy()
)

# Ensure stable dtypes for filtering
clustered_convs["dataset"] = clustered_convs["dataset"].astype(str)
clustered_convs["conv_id"] = pd.to_numeric(clustered_convs["conv_id"], errors="coerce").astype("Int64")
clustered_convs["topic_id"] = pd.to_numeric(clustered_convs["topic_id"], errors="coerce").fillna(-1).astype(int)

# Ensure df_turns_enriched keys are compatible
df_turns_enriched["dataset"] = df_turns_enriched["dataset"].astype(str)
df_turns_enriched["conv_id"] = pd.to_numeric(df_turns_enriched["conv_id"], errors="coerce").astype("Int64")
df_turns_enriched["turn_id"] = pd.to_numeric(df_turns_enriched["turn_id"], errors="coerce").astype("Int64")

sandbox_rows = []

for _, rep in df_repairs.iterrows():
    tid = int(rep["topic_id"])
    tlabel = str(rep.get("topic_label", "") or "").strip()

    # Pick up to 2 example conversations in this topic
    convs = (
        clustered_convs[clustered_convs["topic_id"] == tid]
        .dropna(subset=["conv_id"])
        .head(2)
        .to_dict(orient="records")
    )

    examples = []
    for c in convs:
        ds = str(c["dataset"])
        cid = int(c["conv_id"])

        examples.append({
            "dataset": ds,
            "conv_id": cid,
            "snippet": conversation_snippet(df_turns_enriched, ds, cid),
        })

    repaired_prompt = BASELINE_SYSTEM_PROMPT
    snippet = str(rep.get("system_prompt_snippet", "") or "").strip()
    if snippet:
        repaired_prompt += "\n\n" + snippet

    sandbox_rows.append({
        "topic_id": tid,
        "topic_label": tlabel,
        "baseline_system_prompt": BASELINE_SYSTEM_PROMPT,
        "repaired_system_prompt": repaired_prompt,
        "example_conversations": examples,
        "test_user_messages": normalize_list_field(rep.get("evaluation_checks", [])),
        "guardrail_rules": normalize_list_field(rep.get("guardrail_rules", [])),
    })

sandbox_path = DATA_DASHBOARD / "dashboard_sandbox_cases.json"
export_json(sandbox_path, sandbox_rows)

# Preview first item
sandbox_rows[:1]


Saved: C:\Users\User\Documents\retailmind-prototype\data\dashboard\dashboard_sandbox_cases.json | records: 6


[{'topic_id': 1,
  'topic_label': 'Topic 1: MISSING_CONTEXT / TONE_ISSUE',
  'baseline_system_prompt': 'You are a helpful assistant. Be concise, accurate, and user-focused. If information is missing, ask clarifying questions. Avoid repeating the same question.',
  'repaired_system_prompt': "You are a helpful assistant. Be concise, accurate, and user-focused. If information is missing, ask clarifying questions. Avoid repeating the same question.\n\nEnsure that the bot provides contextually relevant questions and maintains a tone that resonates with the user's mood.",
  'example_conversations': [{'dataset': 'CCPE',
    'conv_id': 80,
    'snippet': "System: What kind of movies do you like, and why do you like this type of movie?\nUser: Comedies, I like to laugh.\nSystem: Can you name a particular movie that you like?\nUser: Liar, Liar\nSystem: Jim Carrey was really funny.\nUser: I agree.\nSystem: Can you give me a couple of reasons why you like this movie?\nUser: I like Jim Carrey in the